# iPad touch feasibility — synthetic canvas only

This notebook never imports LCProp or runs science. Use a hosted Colab session in which ipympl already works. It installs nothing, changes no dependencies, and does not switch an existing controller's backend. If dependencies/backend are missing, stop and use the separately qualified setup workflow.

Test Safari and Chrome separately; record device/iPadOS/browser versions. Compare finger, Pencil and mouse. Do not infer browser acceptance from successful Python initialization.

## Opening this notebook directly in Colab

A newly opened notebook normally has its own runtime; another notebook's setup does not carry over. This diagnostic has no automatic installation cell. For an already configured runtime, skip installation and initialize the backend below only if no controller/figure has yet been created.

For a new runtime, follow the [published LCProp 0.4.0 setup guide](https://github.com/mcroning/LCProp/blob/dfc426a26d288a540228d3ab24c990cf33105b0e/docs/notebook_package.md). In a temporary setup cell in **this notebook**, run the guide's command:

```python
%pip install "lcprop[colab]==0.4.0"
```

Do not add upgrade or force-reinstall flags. After installation, select **Runtime → Restart session** (not disconnect/delete). Before running diagnostic cell 1, use a temporary initialization cell:

```python
from importlib import metadata
assert metadata.version("lcprop") == "0.4.0"
from google.colab import output
output.enable_custom_widget_manager()
%matplotlib widget
```

These are manual prerequisite instructions, not part of the diagnostic or a command to run a solver. Do not construct InteractiveLC or press a scientific Run button. Do not switch a live controller's backend. If setup fails, stop and retain the error; do not patch packages or the browser DOM.

Run numbered diagnostic cells in order. Use **Start trial → gesture → release and wait → End trial**. In the evidence cell, replace the observation placeholders before running it. If no mouse is available, record mouse tests as **not tested** rather than passing. Keep unobservable frontend details marked **UNVERIFIED**. Download `evidence.json` from the Colab Files panel. Cleanup is optional and should follow evidence retention.


In [ ]:
# 1. Record environment; no installation or backend switch
import sys, json, platform, time, uuid, math
from importlib import metadata
import matplotlib
from IPython.display import display
import ipywidgets as W
from google.colab import output
output.enable_custom_widget_manager()
required = ("matplotlib", "ipympl", "ipywidgets", "IPython")
versions = {name: metadata.version(name) for name in required}
backend = matplotlib.get_backend()
if "ipympl" not in backend.lower():
    raise RuntimeError("Stop: this diagnostic requires an already initialized ipympl session. Use the qualified setup first.")
import matplotlib.pyplot as plt
provenance = {"python": sys.version, "platform": platform.platform(),
              "backend": backend, "installed_versions": versions,
              "loaded_versions": {n: getattr(sys.modules.get(n), "__version__", None) for n in required},
              "actual_browser_frontend_version": "UNVERIFIED — fill from browser widget/module evidence, not Python metadata"}
print(json.dumps(provenance, indent=2))


## 2. Disposable canvas and baseline logging

No beam editor is attached: draw a gesture anywhere inside the axes. Python records events only. Press **Start trial** before each gesture; give every trial a distinct label such as `Safari-finger-baseline-1`. Finish with **End trial** after releasing. Wait for comm delivery before ending.

A continuous drag should produce one left-button press, motion with finite in-axes coordinates, then one release. Moving outside axes may produce valid display coordinates but absent data coordinates. Figure-leave is recorded; the ipympl mouse protocol does not expose pointercancel or lostpointercapture, so an interrupted sequence cannot prove which browser event occurred.


In [ ]:
# Rerunning closes only the previous diagnostic figure and its listeners.
if "touch_diag" in globals():
    for cid in touch_diag["connections"]:
        touch_diag["fig"].canvas.mpl_disconnect(cid)
    plt.close(touch_diag["fig"])
    touch_diag["box"].close()
with plt.ioff():
    fig, ax = plt.subplots(figsize=(5, 4))
ax.set(xlim=(0, 100), ylim=(0, 100), xlabel="synthetic x", ylabel="synthetic y",
       title="Touch diagnostic — no scientific execution")
ax.grid(True)
ax.plot([50], [50], "o", markersize=14)
canvas = fig.canvas
canvas.capture_scroll = False
canvas.toolbar_visible = False
canvas.resizable = False
scope = "lcprop-touch-diag-" + uuid.uuid4().hex
canvas.add_class(scope)  # Supported DOMWidget CSS-class API.
style = W.HTML()
mode = W.ToggleButtons(options=["baseline", "scoped CSS"], description="Mode")
label = W.Text(description="Trial", value="Safari-finger-baseline-1")
start = W.Button(description="Start trial")
end = W.Button(description="End trial")
status = W.HTML(value="Ready; baseline has no touch-scroll suppression.")
trials = []
active = None

def scalar(v):
    try:
        v = float(v)
        return v if math.isfinite(v) else None
    except (TypeError, ValueError):
        return None

def record(event):
    if active is None:
        return
    active["events"].append({"n": len(active["events"]), "time": time.monotonic(),
        "name": event.name, "x": scalar(event.x), "y": scalar(event.y),
        "xdata": scalar(event.xdata), "ydata": scalar(event.ydata),
        "inaxes": event.inaxes is ax, "button": str(getattr(event, "button", None)),
        "buttons": sorted(str(b) for b in (getattr(event, "buttons", None) or []))})

def set_mode(change):
    if change["name"] != "value":
        return
    # All rules require the unique class of this one disposable canvas widget.
    # No global scroll suppression, querySelector, scripts, or DOM reparenting.
    style.value = ("<style>." + scope + " canvas {touch-action: none; outline: 3px solid magenta;}</style>"
                   if change["new"] == "scoped CSS" else "")
    status.value = "Mode changed. Release all contacts before starting a new trial."

def begin(_):
    global active
    if active is not None:
        status.value = "End the current trial first."
        return
    active = {"label": label.value, "mode": mode.value, "events": [],
              "started": time.time()}
    trials.append(active)
    mode.disabled = True
    status.value = "Recording. Perform one gesture, release, wait, then End trial."

def finish(_):
    global active
    if active is None:
        return
    active["ended"] = time.time()
    names = [e["name"] for e in active["events"]]
    active["summary"] = {"presses": names.count("button_press_event"),
                         "motions": names.count("motion_notify_event"),
                         "releases": names.count("button_release_event"),
                         "leaves": names.count("figure_leave_event")}
    status.value = "Recorded " + str(active["summary"]) + "; inspect sequence in evidence cell."
    active = None
    mode.disabled = False

mode.observe(set_mode, names="value")
start.on_click(begin)
end.on_click(finish)
connections = [canvas.mpl_connect(n, record) for n in (
    "button_press_event", "motion_notify_event", "button_release_event",
    "figure_enter_event", "figure_leave_event")]
box = W.VBox([style, mode, W.HBox([label, start, end]), status, canvas])
touch_diag = {"fig": fig, "connections": connections, "box": box}
provenance["canvas_class"] = type(canvas).__module__ + "." + type(canvas).__qualname__
provenance["requested_frontend"] = {n: getattr(canvas, n, None) for n in (
    "_model_module", "_model_module_version", "_view_module", "_view_module_version")}
print("Requested module versions are constraints, NOT proof of the actual loaded browser bundle.")
print("CSS scope:", scope)
display(box)


## 3. Comparison and stop condition

1. **Baseline:** finger, Pencil, mouse; repeat each three times. Record notebook scrolling, event order and coordinates.
2. Select **scoped CSS** before contact. A magenta outline should appear on this canvas. If absent, stop the CSS experiment: widget HTML sanitization/framing may prevent the style reaching its target. Do not inject JavaScript or manipulate Colab's DOM.
3. The outline proves rule delivery, not the computed `touch-action` value. Where supported browser inspection is available, record the top canvas's computed `touch-action: none` and the actual loaded `jupyter-matplotlib` module version/URL. Without inspection, mark these **unverified**; changed behavior alone is not proof of frontend identity.
4. Repeat the same gestures. Verify the rest of the notebook still scrolls normally. If it does not, stop and return to baseline.
5. Test release outside the axes/canvas, a second finger during a drag, a browser interruption, and recovery with the next gesture. Record missing terminal events. Do not equate missing release with verified pointer cancellation.
6. Restore **baseline** and repeat one mouse and finger trial. Check there are no doubled press/release pairs. Python logs cannot distinguish delayed compatibility mouse events from native input without browser evidence.

No private selectors or frame traversal are permitted. If style delivery fails, report the limitation. Do not proceed to an adapter in this notebook.


In [ ]:
# 4. Fill observations, then print and save local evidence (no network publication).
from pathlib import Path
import tempfile
observations = {
    "device": "FILL", "ipados": "FILL", "browser_version": "FILL",
    "input_devices": "FILL", "notebook_url": "FILL",
    "actual_frontend_version_and_evidence": "UNVERIFIED",
    "magenta_outline_scoped_only": "UNTESTED",
    "computed_touch_action": "UNVERIFIED",
    "outside_notebook_scroll": "UNTESTED",
    "finger_baseline_vs_css": "UNTESTED",
    "pencil_baseline_vs_css": "UNTESTED",
    "mouse_baseline_vs_css": "UNTESTED",
    "cancel_outside_interrupt_and_recovery": "UNTESTED",
    "duplicate_event_observations": "UNTESTED",
}
if active is not None:
    raise RuntimeError("End the active trial before exporting.")
evidence = {"provenance": provenance, "observations": observations, "trials": trials,
            "scientific_execution": False}
folder = Path(tempfile.mkdtemp(prefix="lcprop-touch-5a-"))
path = folder / "evidence.json"
path.write_text(json.dumps(evidence, indent=2, allow_nan=False))
print(json.dumps(evidence, indent=2, allow_nan=False))
print("Retain this file using Colab Files → Download:", path)


## Cleanup

After retaining evidence, the optional cell below removes the scoped style and closes only this diagnostic. Existing LCProp controllers are untouched.


In [ ]:
style.value = ""
for cid in touch_diag["connections"]:
    fig.canvas.mpl_disconnect(cid)
plt.close(fig)
box.close()
